# 체성분 결과지 → 수치 추출 모델 비교 (Main Quest 2 PoC)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jsseo0420/ai-agent-mainquest2-inbody-routine-poc/blob/main/notebooks/poc.ipynb)

가상 체성분 결과지 15장에서 **핵심 8개 수치**를 뽑아, 세 가지 방식을 비교한다.

| 방식 | 설명 | 비용 | 사진이 외부로 나가나 |
|---|---|---|---|
| A. EasyOCR + 규칙 | 글자를 읽은 뒤, 라벨 오른쪽에 있는 숫자를 규칙으로 찾는다 (기존 방식에 가까운 기준선) | 무료 | 아니오 |
| B. Gemini Flash (API) | 이미지를 주고 JSON 스키마로 수치를 받는다 | 무료 등급 | 예 (Google) |
| C. Qwen2.5-VL 3B (로컬) | 오픈소스 비전 모델을 Colab GPU에서 직접 실행한다 | 무료 | 아니오 |
| D. Claude 비전 (선택) | 유료 API라 기본 실행에서는 키가 없으면 건너뛴다 | 유료 | 예 (Anthropic) |

**비교 항목**: 필드 정확도, 촬영 상태(깨끗함·기울기·흐림·그림자·저해상도)별 정확도, 장당 처리 시간, 장당 비용

### 실행 전 준비 (Colab)
1. **런타임 → 런타임 유형 변경 → GPU(T4)** 를 켠다. A와 C가 GPU를 쓴다.
2. 왼쪽 **🔑 Secrets** 메뉴에서 `GEMINI_API_KEY`를 추가하고 **노트북 액세스**를 켠다. (`ANTHROPIC_API_KEY`는 선택)
3. 키가 없는 방식은 자동으로 건너뛴다. 키를 셀에 직접 적지 않는다.
4. 결과는 `results/`에 저장되어, 다시 실행해도 API를 또 부르지 않는다 (`FORCE_RERUN = True`로 바꾸면 다시 실행).

> 결과지는 모두 가상 데이터이며, 이 노트북의 출력은 의료 조언이 아니다.

In [ ]:
# 저장소 받기 + 패키지 설치 (로컬에서 저장소 안에서 실행하면 clone을 건너뛴다)
import os
import subprocess

REPO_URL = "https://github.com/jsseo0420/ai-agent-mainquest2-inbody-routine-poc.git"
if not os.path.exists("data/ground_truth.json"):
    if not os.path.exists("poc_repo"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "poc_repo"], check=True)
    os.chdir("poc_repo")
print("작업 폴더:", os.getcwd())

!pip install -q easyocr google-genai anthropic pandas accelerate

In [ ]:
import base64, json, re, time
from pathlib import Path

import pandas as pd
from IPython.display import Image as IPImage, display


def get_secret(name):
    """Colab Secrets → 환경변수 순서로 키를 찾는다. 값은 출력하지 않는다."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY")
print("Gemini 키:", "있음" if GEMINI_API_KEY else "없음 → B 건너뜀")
print("Claude 키:", "있음" if ANTHROPIC_API_KEY else "없음 → C 건너뜀")

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
FORCE_RERUN = False  # True면 저장된 결과를 무시하고 API를 다시 호출한다

## 1. 데이터와 채점 기준

- 정답: `data/ground_truth.json` (생성기가 결과지를 만들 때 함께 저장한 값)
- **필드가 맞았다** = 결과지에 찍힌 자릿수로 반올림했을 때 정답과 같다 (예: 체중은 소수 1자리, 복부지방률은 2자리)
- 값을 못 찾으면(`null`) 틀린 것으로 센다

In [ ]:
truth = json.loads(Path("data/ground_truth.json").read_text(encoding="utf-8"))
FIELDS = list(truth["fields"].keys())
FIELD_NAMES = truth["fields"]
DECIMALS = {"weight_kg": 1, "skeletal_muscle_kg": 1, "body_fat_kg": 1, "body_fat_pct": 1,
            "bmi": 1, "bmr_kcal": 0, "score": 0, "whr": 2}
SAMPLES = truth["samples"]
print(f"결과지 {len(SAMPLES)}장, 필드 {len(FIELDS)}개")
print(pd.Series([s["variant"] for s in SAMPLES]).value_counts().to_string())
display(IPImage(filename=f"data/samples/{SAMPLES[0]['file']}", width=420))


def is_correct(field, pred, answer):
    if pred is None:
        return False
    try:
        return round(float(pred), DECIMALS[field]) == round(float(answer), DECIMALS[field])
    except (TypeError, ValueError):
        return False


def run_method(name, extract_fn):
    """extract_fn(image_path) -> (fields dict, info dict). 결과를 results/{name}.json에 저장·재사용한다.
    저장된 결과 중 오류가 난 장만 다시 실행한다 (API 오류를 고친 뒤 셀만 다시 돌리면 된다)."""
    cache = RESULTS / f"{name}.json"
    saved = {}
    if cache.exists() and not FORCE_RERUN:
        saved = {r["file"]: r for r in json.loads(cache.read_text(encoding="utf-8")) if r["error"] is None}
        print(f"[{name}] 저장된 결과 {len(saved)}장 재사용, {len(SAMPLES) - len(saved)}장 실행")
    records = []
    for s in SAMPLES:
        if s["file"] in saved:
            records.append(saved[s["file"]])
            continue
        path = f"data/samples/{s['file']}"
        start = time.perf_counter()
        try:
            pred, info = extract_fn(path)
            error = None
        except Exception as e:  # 한 장이 실패해도 나머지는 계속한다
            pred, info, error = {}, {}, f"{type(e).__name__}: {e}"
        # 한도 때문에 기다린 시간은 빼고, 실제 처리에 걸린 시간만 기록한다
        elapsed = info.get("api_seconds", time.perf_counter() - start)
        pred = {f: pred.get(f) for f in FIELDS}
        correct = {f: is_correct(f, pred[f], s["fields"][f]) for f in FIELDS}
        records.append({"file": s["file"], "variant": s["variant"], "seconds": round(elapsed, 2),
                        "pred": pred, "correct": correct, "info": info, "error": error})
        print(f"[{name}] {s['file']}: {sum(correct.values())}/{len(FIELDS)}  {elapsed:.1f}s"
              + (f"  오류: {error}" if error else ""))
    cache.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")
    return records


ALL_RESULTS = {}

## 2. 방식 A: EasyOCR + 규칙 (기준선)

사람이 결과지를 보고 옮겨 적는 과정을 규칙으로 흉내 낸다.
글자 상자마다 위치가 나오므로, **라벨(예: "골격근량") 오른쪽 같은 줄에 있는 가장 가까운 숫자**를 값으로 본다.
점수는 "/ 100점" 왼쪽 숫자, 복부지방률은 제목 아래에서 가장 큰 `0.xx` 숫자를 쓴다.
표준범위·눈금 숫자와 섞이기 쉬워, 이 규칙이 어디서 틀리는지가 비교의 핵심이다.

In [ ]:
import easyocr

reader = easyocr.Reader(["ko", "en"], gpu=True)  # GPU가 없으면 자동으로 CPU를 쓴다 (느림)

NUM_RE = re.compile(r"^-?\d{1,4}(?:[.,]\d+)*$")


def ocr_boxes(path):
    boxes = []
    for bbox, text, conf in reader.readtext(path, detail=1):
        xs = [p[0] for p in bbox]
        ys = [p[1] for p in bbox]
        boxes.append({"text": text, "flat": re.sub(r"\s+", "", text), "conf": conf,
                      "x0": min(xs), "x1": max(xs), "y0": min(ys), "y1": max(ys),
                      "cx": sum(xs) / 4, "cy": sum(ys) / 4, "h": max(ys) - min(ys)})
    return boxes


def to_number(text):
    """'1,385kcal' → 1385, '75.8kg' → 75.8, 숫자가 아니면 None"""
    t = re.sub(r"(kg|kcal|cm|%|점|L)$", "", re.sub(r"\s+", "", text), flags=re.I)
    if not NUM_RE.match(t):
        return None
    if "," in t and re.match(r"^\d{1,3}(,\d{3})+$", t):
        t = t.replace(",", "")
    t = t.replace(",", ".")
    try:
        return float(t)
    except ValueError:
        return None


def right_of(label, boxes):
    """라벨과 같은 줄, 오른쪽에 있는 가장 가까운 숫자"""
    best = None
    for b in boxes:
        v = to_number(b["text"])
        if v is None or b["x0"] < label["x1"] - 5:
            continue
        if abs(b["cy"] - label["cy"]) > max(label["h"], b["h"]) * 0.7:
            continue
        dx = b["x0"] - label["x1"]
        if best is None or dx < best[0]:
            best = (dx, v)
    return None if best is None else best[1]


def find(boxes, pred):
    return [b for b in boxes if pred(b["flat"])]


def first_value(boxes, label_preds):
    """우선순위대로 라벨 조건을 시도해 처음 찾은 값"""
    for pred in label_preds:
        for label in find(boxes, pred):
            v = right_of(label, boxes)
            if v is not None:
                return v
    return None


def extract_ocr(path):
    boxes = ocr_boxes(path)
    out = {
        "weight_kg": first_value(boxes, [lambda t: t.startswith("체중(kg"), lambda t: t == "체중"]),
        "skeletal_muscle_kg": first_value(boxes, [lambda t: t.startswith("골격근량")]),
        "body_fat_kg": first_value(boxes, [lambda t: t.startswith("체지방량(kg"), lambda t: t.startswith("체지방량")]),
        "body_fat_pct": first_value(boxes, [lambda t: t.startswith("체지방률") and "~" not in t]),
        "bmi": first_value(boxes, [lambda t: t.upper().startswith("BMI") and "~" not in t]),
        "bmr_kcal": None,
        "score": None,
        "whr": None,
    }
    # 기초대사량: 콤마·마침표가 섞여 읽혀도 숫자만 남긴다
    for label in find(boxes, lambda t: t.startswith("기초대사량")):
        for b in boxes:
            if b["x0"] > label["x1"] - 5 and abs(b["cy"] - label["cy"]) < max(label["h"], b["h"]) * 0.7:
                digits = re.sub(r"\D", "", b["text"])
                if 3 <= len(digits) <= 4:
                    out["bmr_kcal"] = int(digits)
                    break
    # 점수: "65/100점"처럼 붙어 읽히거나, "/100점" 상자 왼쪽에 따로 읽힌다
    for b in boxes:
        m = re.search(r"(\d{2,3})/100", b["flat"])
        if m:
            out["score"] = int(m.group(1))
            break
    if out["score"] is None:
        for mark in find(boxes, lambda t: "100점" in t):
            cands = [b for b in boxes if to_number(b["text"]) is not None and b["x1"] <= mark["x0"] + 5
                     and abs(b["cy"] - mark["cy"]) < b["h"]]
            if cands:
                out["score"] = int(to_number(max(cands, key=lambda b: b["h"])["text"]))
                break
    # 복부지방률: 제목 아래 0.xx 숫자 중 글자가 가장 큰 것 (눈금 0.70 등은 작다)
    for title in find(boxes, lambda t: t.startswith("복부지방률")):
        cands = [b for b in boxes if re.match(r"^[01][.,]\d{2}$", b["flat"]) and b["cy"] > title["cy"]
                 and b["cy"] - title["cy"] < title["h"] * 8 and abs(b["cx"] - title["cx"]) < 200]
        if cands:
            out["whr"] = to_number(max(cands, key=lambda b: b["h"])["flat"])
            break
    return out, {"boxes": len(boxes)}


ALL_RESULTS["A_easyocr_rules"] = run_method("A_easyocr_rules", extract_ocr)

## 3. LLM 방식 공통: 프롬프트와 출력 스키마

두 비전 LLM에 **같은 프롬프트와 같은 JSON 스키마**를 준다.
헷갈리는 숫자(표준범위, 그래프 눈금, 적정체중·조절값, 기초대사량 표준범위)는 쓰지 말라고 명시하고,
읽을 수 없는 값은 지어내지 말고 `null`로 달라고 한다.

In [ ]:
PROMPT = """이 이미지는 체성분 분석 결과지 사진이다. 아래 8개 항목의 '측정치'만 숫자로 추출해라.

- weight_kg: 체중 (kg)
- skeletal_muscle_kg: 골격근량 (kg)
- body_fat_kg: 체지방량 (kg)
- body_fat_pct: 체지방률 (%)
- bmi: BMI
- bmr_kcal: 기초대사량 (kcal, 정수)
- score: 체성분 점수 (100점 만점 중 점수, 정수)
- whr: 복부지방률 (소수 둘째 자리)

규칙:
- 표준범위, 그래프 눈금, 적정체중, 체중·지방·근육 조절값, 기초대사량의 (표준) 범위는 측정치가 아니므로 쓰지 않는다.
- 결과지에 찍힌 자릿수 그대로 쓰고, 단위는 빼고 숫자만 쓴다.
- 흐리거나 가려져 확실히 읽을 수 없는 항목은 추측하지 말고 null로 둔다."""

FIELD_SCHEMA = {
    "type": "object",
    "properties": {f: {"anyOf": [{"type": "number"}, {"type": "null"}]} for f in FIELDS},
    "required": FIELDS,
    "additionalProperties": False,
}


def image_b64(path):
    return base64.standard_b64encode(Path(path).read_bytes()).decode("utf-8")

## 4. 방식 B: Gemini Flash

`google-genai` SDK로 이미지와 프롬프트를 보내고, `response_schema`로 JSON을 받는다.

- Gemini 모델은 자주 바뀐다. `gemini-2.5-flash`는 새 사용자에게 404(더 이상 제공 안 함)가 나와서, API가 안내한 `gemini-3.8-flash`를 쓴다.
  셀을 실행하면 이 키로 쓸 수 있는 Flash 모델 목록을 먼저 보여 준다. 404가 나면 목록의 이름으로 `GEMINI_MODEL`을 바꾼다.
- **무료 등급으로 실행하므로 비용은 0으로 기록한다.** 유료 등급으로 비교하려면
  [가격 페이지](https://ai.google.dev/gemini-api/docs/pricing)의 값을 `GEMINI_PRICE`에 넣는다 (토큰 수는 항상 기록한다).
- 무료 등급은 **분당 5회** 한도가 있고, 서버가 바쁘면 503을 돌려준다.
  그래서 호출 사이에 13초 간격을 두고, 429·503이 나면 기다렸다가 최대 5번까지 다시 시도한다.
  15장 처리에 4분 정도 걸리며, 비교표의 시간에는 기다린 시간을 빼고 **실제 응답 시간만** 기록한다.

In [ ]:
GEMINI_MODEL = "gemini-3.8-flash"
GEMINI_PRICE = None  # 유료 등급일 때만: {"input": 입력가, "output": 출력가}  (USD / 100만 토큰)
GEMINI_MIN_INTERVAL = 13  # 초. 무료 등급 분당 5회 한도에 맞춘 호출 간격
GEMINI_MAX_TRIES = 5

if GEMINI_API_KEY:
    from typing import Optional
    from google import genai
    from google.genai import types
    from pydantic import BaseModel

    class BodyComposition(BaseModel):
        weight_kg: Optional[float]
        skeletal_muscle_kg: Optional[float]
        body_fat_kg: Optional[float]
        body_fat_pct: Optional[float]
        bmi: Optional[float]
        bmr_kcal: Optional[float]
        score: Optional[float]
        whr: Optional[float]

    gemini = genai.Client(api_key=GEMINI_API_KEY)

    # 이 키로 쓸 수 있는 Flash 모델 이름 (모델 이름이 바뀌었을 때 참고용)
    try:
        flash = sorted(m.name.removeprefix("models/") for m in gemini.models.list() if "flash" in m.name)
        print("사용 가능한 Flash 모델:", ", ".join(flash) or "(없음)")
    except Exception as e:
        print("모델 목록을 가져오지 못함:", e)

    from google.genai import errors as genai_errors

    _last_call = [0.0]

    def call_gemini(path):
        """한도(429)·서버 혼잡(503)이면 기다렸다가 다시 시도한다. (응답, 실제 응답 시간)을 돌려준다."""
        for attempt in range(1, GEMINI_MAX_TRIES + 1):
            wait = GEMINI_MIN_INTERVAL - (time.monotonic() - _last_call[0])
            if wait > 0:
                time.sleep(wait)
            _last_call[0] = time.monotonic()
            start = time.perf_counter()
            try:
                resp = gemini.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=[types.Part.from_bytes(data=Path(path).read_bytes(), mime_type="image/jpeg"), PROMPT],
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        response_schema=BodyComposition,
                        temperature=0,
                    ),
                )
                return resp, time.perf_counter() - start
            except genai_errors.APIError as e:
                if e.code not in (429, 503) or attempt == GEMINI_MAX_TRIES:
                    raise
                m = re.search(r"retry in ([\d.]+)s", str(e))
                delay = float(m.group(1)) + 1 if m else 15 * attempt
                print(f"    {e.code} → {delay:.0f}초 기다렸다가 다시 시도 ({attempt}/{GEMINI_MAX_TRIES})")
                time.sleep(delay)

    def extract_gemini(path):
        resp, api_seconds = call_gemini(path)
        usage = resp.usage_metadata
        in_tok = usage.prompt_token_count or 0
        out_tok = (usage.candidates_token_count or 0) + (getattr(usage, "thoughts_token_count", 0) or 0)
        cost = 0 if GEMINI_PRICE is None else (in_tok * GEMINI_PRICE["input"] + out_tok * GEMINI_PRICE["output"]) / 1_000_000
        return json.loads(resp.text), {"input_tokens": in_tok, "output_tokens": out_tok, "cost_usd": cost,
                                       "model": GEMINI_MODEL, "api_seconds": api_seconds}

    ALL_RESULTS["B_gemini_flash"] = run_method("B_gemini_flash", extract_gemini)
else:
    print("GEMINI_API_KEY가 없어 건너뜀")

## 5. 방식 C: Qwen2.5-VL 3B (Colab에서 직접 실행)

오픈소스 비전 언어 모델 [Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)를 Colab GPU에 올려 돌린다.

- **API 키·비용이 없고, 결과지 사진이 외부 서버로 나가지 않는다.** 건강 정보를 다루는 서비스에서 중요한 장점이다.
- T4(16GB)에 맞게 float16으로 올린다 (약 8GB). 처음 실행 때 모델을 받는 데 몇 분 걸린다.
- API처럼 JSON 스키마를 강제할 수 없어서, 답에서 `{...}` 부분을 찾아 파싱한다. 파싱에 실패하면 그 장은 오류로 센다.
- 라이선스: 3B 모델은 Qwen Research License(비상업적 연구 용도)다. 상업 서비스로 가려면 다른 모델이나 라이선스를 검토해야 한다.

In [ ]:
QWEN_MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
RUN_LOCAL_VLM = True  # GPU가 없거나 시간을 아끼려면 False

import torch

if RUN_LOCAL_VLM and torch.cuda.is_available():
    from PIL import Image
    from transformers import AutoProcessor
    try:
        from transformers import Qwen2_5_VLForConditionalGeneration as QwenVL
    except ImportError:  # 최신 transformers에서는 이 이름을 쓴다
        from transformers import AutoModelForImageTextToText as QwenVL

    qwen = QwenVL.from_pretrained(QWEN_MODEL, torch_dtype=torch.float16, device_map="cuda")
    # 이미지 토큰 수를 제한해 T4 메모리와 속도를 맞춘다 (28x28 패치 기준 약 1,280개)
    qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL, min_pixels=256 * 28 * 28, max_pixels=1280 * 28 * 28)

    QWEN_PROMPT = PROMPT + "\n\n반드시 아래 키를 가진 JSON 객체 하나만 출력해라. 설명은 쓰지 마라.\n" + json.dumps(
        {f: "숫자 또는 null" for f in FIELDS}, ensure_ascii=False)

    def parse_json_answer(text):
        m = re.search(r"\{.*\}", text, re.S)
        if not m:
            raise ValueError(f"JSON을 찾지 못함: {text[:80]!r}")
        data = json.loads(m.group(0))
        return {f: (data.get(f) if isinstance(data.get(f), (int, float)) else to_number(str(data.get(f))))
                for f in FIELDS}

    def extract_qwen(path):
        image = Image.open(path).convert("RGB")
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": QWEN_PROMPT}]}]
        text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = qwen_processor(text=[text], images=[image], return_tensors="pt").to(qwen.device)
        with torch.no_grad():
            out_ids = qwen.generate(**inputs, max_new_tokens=300, do_sample=False)
        answer = qwen_processor.batch_decode(out_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        return parse_json_answer(answer), {"cost_usd": 0, "raw": answer[:300],
                                           "input_tokens": int(inputs["input_ids"].shape[1])}

    ALL_RESULTS["C_qwen25vl_3b_local"] = run_method("C_qwen25vl_3b_local", extract_qwen)
else:
    print("GPU가 없거나 RUN_LOCAL_VLM=False라 건너뜀")

## 6. (선택) 방식 D: Claude 비전

유료 API라 기본 실행에서는 `ANTHROPIC_API_KEY`가 없으면 건너뛴다. 크레딧이 있을 때 같은 조건으로 비교할 수 있게 코드를 남겨 둔다.

`anthropic` SDK로 이미지(base64)와 프롬프트를 보내고, `output_config.format`(JSON 스키마)으로 받는다.

- 모델: `claude-opus-5-5` (입력 $4 / 출력 $20, 100만 토큰당)
- 단순 추출 작업이라 `effort`는 `low`로 둔다. 이 모델은 생각(thinking)을 끌 수 없고, effort로 깊이를 조절한다.
- 안전 분류기가 요청을 거절하는 경우를 대비해 **서버 측 대체 모델(fallbacks)**을 켜 두었다. 거절되면 같은 요청을 다른 모델이 이어서 처리한다.

In [ ]:
CLAUDE_MODEL = "claude-opus-5-5"
CLAUDE_PRICE = {"input": 4.00, "output": 20.00}  # USD / 100만 토큰

if ANTHROPIC_API_KEY:
    import anthropic

    claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    def extract_claude(path):
        resp = claude.beta.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=2000,
            betas=["server-side-fallback-2026-07-01"],
            fallbacks="default",
            output_config={"effort": "low", "format": {"type": "json_schema", "schema": FIELD_SCHEMA}},
            messages=[{
                "role": "user",
                "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_b64(path)}},
                    {"type": "text", "text": PROMPT},
                ],
            }],
        )
        if resp.stop_reason == "refusal":
            raise RuntimeError(f"거절됨: {getattr(resp.stop_details, 'category', None)}")
        text = next(b.text for b in resp.content if b.type == "text")
        u = resp.usage
        cost = (u.input_tokens * CLAUDE_PRICE["input"] + u.output_tokens * CLAUDE_PRICE["output"]) / 1_000_000
        return json.loads(text), {"input_tokens": u.input_tokens, "output_tokens": u.output_tokens,
                                  "cost_usd": cost, "model": resp.model}

    ALL_RESULTS["D_claude_vision"] = run_method("D_claude_vision", extract_claude)
else:
    print("ANTHROPIC_API_KEY가 없어 건너뜀")

## 7. 결과 비교

- **필드 정확도**: 15장 × 8필드 중 맞힌 비율
- **완벽한 장 비율**: 8개를 모두 맞힌 결과지 비율 (사용자가 고칠 필요가 없는 경우)
- **장당 시간·비용**: 결과지 한 장을 처리하는 데 걸린 시간과 API 비용
- 기존 방식(수기 입력)은 문제 정의서 기준 **약 2분/장, 비용 0**

In [ ]:
def summarize(results):
    rows = []
    for name, recs in results.items():
        cells = [c for r in recs for c in r["correct"].values()]
        costs = [r["info"].get("cost_usd", 0) or 0 for r in recs]
        rows.append({
            "방식": name,
            "필드 정확도": f"{sum(cells) / len(cells):.1%}",
            "완벽한 장": f"{sum(all(r['correct'].values()) for r in recs)}/{len(recs)}",
            "장당 시간(초)": round(sum(r["seconds"] for r in recs) / len(recs), 2),
            "장당 비용(USD)": round(sum(costs) / len(recs), 5),
            "15장 비용(USD)": round(sum(costs), 4),
            "오류": sum(r["error"] is not None for r in recs),
        })
    return pd.DataFrame(rows).set_index("방식")


summary = summarize(ALL_RESULTS)
summary.to_csv(RESULTS / "summary.csv", encoding="utf-8-sig")
summary

In [ ]:
# 촬영 상태별 필드 정확도
by_variant = pd.DataFrame({
    name: pd.DataFrame([{"variant": r["variant"], "acc": sum(r["correct"].values()) / len(FIELDS)} for r in recs])
            .groupby("variant")["acc"].mean()
    for name, recs in ALL_RESULTS.items()
}).reindex(["clean", "tilt", "blur", "shadow", "lowres", "combo"])
by_variant.to_csv(RESULTS / "by_variant.csv", encoding="utf-8-sig")
by_variant.style.format("{:.0%}")

In [ ]:
# 필드별 정확도
by_field = pd.DataFrame({
    name: {FIELD_NAMES[f]: sum(r["correct"][f] for r in recs) / len(recs) for f in FIELDS}
    for name, recs in ALL_RESULTS.items()
})
by_field.to_csv(RESULTS / "by_field.csv", encoding="utf-8-sig")
by_field.style.format("{:.0%}")

## 8. 실패 사례

틀린 필드를 모두 모아 본다. 검증 문서(`docs/evaluation.md`)의 실패 사례 3건은 여기서 고른다.

In [ ]:
truth_by_file = {s["file"]: s["fields"] for s in SAMPLES}
failures = pd.DataFrame([
    {"방식": name, "파일": r["file"], "상태": r["variant"], "필드": FIELD_NAMES[f],
     "정답": truth_by_file[r["file"]][f], "추출값": r["pred"][f]}
    for name, recs in ALL_RESULTS.items() for r in recs for f in FIELDS if not r["correct"][f]
])
failures.to_csv(RESULTS / "failures.csv", index=False, encoding="utf-8-sig")
print(f"틀린 필드 {len(failures)}개")
failures

In [ ]:
# 실패가 가장 많은 결과지를 눈으로 확인
if len(failures):
    worst = failures["파일"].value_counts().index[0]
    print(worst, failures[failures["파일"] == worst]["방식"].value_counts().to_dict())
    display(IPImage(filename=f"data/samples/{worst}", width=480))

## 다음 단계

- 비교 결과로 추출 모델을 고르고 근거를 `docs/model_comparison.md`에 정리한다.
- 고른 모델의 출력 JSON을 받아 **헬스도감 운동 목록(`data/exercises.json`) 안에서만** 루틴을 추천하는 단계를 이어서 만든다.
- `results/` 폴더(요약·실패 사례 CSV, 원본 예측 JSON)를 저장소에 커밋하면 결과를 재현·검토할 수 있다.